# Manuka-0527 CPU Prepare

Run this notebook in a CPU Colab runtime. It reads the raw zip, builds plain single-turn pairs, trains the SentencePiece tokenizer, tokenizes examples, and saves prepared artifacts to Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# Run once per fresh Colab runtime if sentencepiece is missing.
try:
    import sentencepiece  # noqa: F401
except ImportError:
    %pip -q install sentencepiece


In [ ]:
from pathlib import Path

DATA_ROOT = Path('/content/drive/MyDrive/text_dataset/manuka/manuka-0527')
PREP_DIR = DATA_ROOT / 'prepared'
MODEL_DIR = DATA_ROOT / 'manuka-model'

# Keep raw data under the shared dataset folder; DATA_ROOT is used for saved prepared/model artifacts.
RAW_ZIP_PATH = Path('/content/drive/MyDrive/text_dataset/dataset/일상대화_데이터.zip')
RAW_ZIP_CANDIDATES = [RAW_ZIP_PATH]
for folder in [
    DATA_ROOT / 'dataset',
    DATA_ROOT,
    Path('/content/drive/MyDrive/text_dataset/dataset'),
]:
    if folder.exists():
        RAW_ZIP_CANDIDATES.extend(sorted(folder.glob('*.zip')))

seen = set()
RAW_ZIP_CANDIDATES = [p for p in RAW_ZIP_CANDIDATES if not (str(p) in seen or seen.add(str(p)))]
ZIP_PATH = next((str(p) for p in RAW_ZIP_CANDIDATES if p.exists()), None)
if ZIP_PATH is None:
    searched = '\n'.join(str(p) for p in RAW_ZIP_CANDIDATES)
    raise FileNotFoundError(
        'Could not find the configured raw dataset zip or any fallback zip. '
        f'Checked:\n{searched or "<no zip candidates found>"}'
    )

PREP_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print('DATA_ROOT:', DATA_ROOT)
print('ZIP_PATH:', ZIP_PATH)
print('PREP_DIR:', PREP_DIR)
print('MODEL_DIR:', MODEL_DIR)
if Path(ZIP_PATH) != RAW_ZIP_PATH:
    print('WARNING: Using fallback zip, not the configured raw dataset zip.')


DATA_ROOT: /content/drive/MyDrive/text_dataset/manuka/manuka-0527
ZIP_PATH: /content/drive/MyDrive/text_dataset/dataset/일상대화_데이터.zip
PREP_DIR: /content/drive/MyDrive/text_dataset/manuka/manuka-0527/prepared
MODEL_DIR: /content/drive/MyDrive/text_dataset/manuka/manuka-0527/manuka-model


# Data Load


In [ ]:
# === Plain single-turn pairs from the full speech corpus ===
import zipfile, json, io, re, random
from collections import Counter

# Keep the task simple for the tiny demo model: previous utterance/span -> next utterance/span.
TEXT_FIELD_PRIORITY = ("form", "original_form")
MIN_LEN = 1
MAX_TEXT_CHARS = 512
MAX_PROMPT_CHARS = 512
MAX_RESPONSE_CHARS = 512
CONTEXT_LENGTH = 512
MAX_LEN = CONTEXT_LENGTH
SHUFFLE, SEED = True, 42
WINDOW_STEPS = [1, 2]
ALLOW_SAME_SPEAKER_PAIRS = True
SPLIT_SENTENCES = True

SENT_RGX = re.compile(r'(?<=[\.!?…])\s+')
SPACE_RGX = re.compile(r"\s+")
KOR_RGX = re.compile(r"[가-힣]")

rng = random.Random(SEED)
examples = []
kind_counts = Counter()
n_files = 0
n_docs = 0
n_utterances = 0
n_usable_units = 0


def clean_text(s: str) -> str:
    s = (s or "").replace("\u200b", " ").replace("\ufeff", " ").strip()
    return SPACE_RGX.sub(" ", s)


def is_usable_text(s: str) -> bool:
    if len(s) < MIN_LEN:
        return False
    return len(KOR_RGX.findall(s)) >= max(1, int(0.10 * len(s)))


def pick_utterance_text(utt: dict) -> str:
    for field in TEXT_FIELD_PRIORITY:
        text = clean_text(utt.get(field, ""))
        if text:
            return text
    return ""


def split_text(s: str):
    s = clean_text(s)
    if not s:
        return []
    parts = [p.strip() for p in SENT_RGX.split(s) if p.strip()] if SPLIT_SENTENCES else [s]
    out = []
    for part in parts or [s]:
        if len(part) <= MAX_TEXT_CHARS:
            out.append(part)
            continue
        cur, cur_len = [], 0
        for word in part.split():
            add = len(word) + (1 if cur else 0)
            if cur and cur_len + add > MAX_TEXT_CHARS:
                out.append(" ".join(cur))
                cur, cur_len = [word], len(word)
            else:
                cur.append(word)
                cur_len += add
        if cur:
            out.append(" ".join(cur))
    return [clean_text(x) for x in out if is_usable_text(clean_text(x))]


def add_example(prompt: str, response: str, kind: str, doc_id: str = ""):
    prompt = clean_text(prompt)[:MAX_PROMPT_CHARS]
    response = clean_text(response)[:MAX_RESPONSE_CHARS]
    if not prompt or not response:
        return
    if not is_usable_text(prompt) or not is_usable_text(response):
        return
    examples.append({"prompt": prompt, "response": response, "kind": kind, "doc_id": doc_id})
    kind_counts[kind] += 1


with zipfile.ZipFile(ZIP_PATH) as zf:
    for name in zf.namelist():
        if not name.lower().endswith(".json"):
            continue
        n_files += 1
        with zf.open(name) as f:
            data = json.load(io.TextIOWrapper(f, encoding="utf-8"))

        for doc in data.get("document", []):
            n_docs += 1
            doc_id = doc.get("id", "")
            utts = sorted(doc.get("utterance", []), key=lambda u: (u.get("start", float("inf")), u.get("id", "")))
            units = []
            for utt in utts:
                n_utterances += 1
                speaker = utt.get("speaker_id") or "unknown"
                for text in split_text(pick_utterance_text(utt)):
                    units.append({"text": text, "speaker": speaker})

            n_usable_units += len(units)
            if len(units) < 2:
                continue

            for step in WINDOW_STEPS:
                for i in range(len(units) - step):
                    left, right = units[i], units[i + step]
                    if ALLOW_SAME_SPEAKER_PAIRS or left["speaker"] != right["speaker"]:
                        add_example(left["text"], right["text"], f"next_{step}", doc_id)

if SHUFFLE:
    rng.shuffle(examples)

pairs = [(ex["prompt"], ex["response"]) for ex in examples]

print(f"JSON files read: {n_files}")
print(f"Documents read: {n_docs}")
print(f"Utterances seen: {n_utterances}")
print(f"Usable text units: {n_usable_units}")
print(f"Training examples: {len(examples)}")
print("Example mix:", dict(kind_counts))
print("Sample:", pairs[0] if pairs else ("<empty>", "<empty>"))

JSON files read: 14229
Documents read: 14229
Utterances seen: 4570823
Usable text units: 4670775
Training examples: 9298863
Example mix: {'next_1': 4656546, 'next_2': 4642317}
Sample: ('그리고 많이 보고 거기에 많은 것들이 나와 있고 네 말대로.', '불안하더라고.')


# Tokenizer


In [ ]:
# SentencePiece tokenizer trained on the same plain pair format used by the model
import os
import sentencepiece as spm

VOCAB_TARGET = 16000
TOKENIZER_PREFIX = "spm16k"

with open("corpus.txt", "w", encoding="utf-8") as f:
    for q, a in pairs:
        f.write(q.replace("\n", " ") + " <sep> " + a.replace("\n", " ") + "\n")

spm.SentencePieceTrainer.train(
    input="corpus.txt",
    model_prefix=TOKENIZER_PREFIX,
    vocab_size=VOCAB_TARGET,
    model_type="unigram",
    character_coverage=0.9998,
    pad_id=0, bos_id=1, eos_id=3, unk_id=4,
    control_symbols=["<sep>"],
    byte_fallback=True,
    hard_vocab_limit=False,
    num_threads=max(1, os.cpu_count() or 1),
)

sp = spm.SentencePieceProcessor(model_file=f"{TOKENIZER_PREFIX}.model")

PAD = sp.pad_id(); BOS = sp.bos_id(); EOS = sp.eos_id(); UNK = sp.unk_id()
SEP = sp.piece_to_id("<sep>")
VOCAB_SIZE = sp.get_piece_size()
SPECIALS = {"PAD": PAD, "BOS": BOS, "SEP": SEP, "EOS": EOS, "UNK": UNK}


def encode_text(s: str):
    return sp.encode(clean_text(s), out_type=int)


def decode_text(ids):
    drop = {PAD, BOS, SEP, EOS, UNK}
    return sp.decode([int(i) for i in ids if int(i) not in drop])

print("Tokenizer vocab size:", VOCAB_SIZE, SPECIALS)

Tokenizer vocab size: 16000 {'PAD': 0, 'BOS': 1, 'SEP': 2, 'EOS': 3, 'UNK': 4}


# Tokenized Examples


In [ ]:
def iter_labeled_examples(examples):
    for ex in examples:
        prompt_ids = encode_text(ex["prompt"])
        response_ids = encode_text(ex["response"]) + [EOS]

        max_response_tokens = max(8, MAX_LEN - 32)
        if len(response_ids) > max_response_tokens:
            response_ids = response_ids[:max_response_tokens]
            response_ids[-1] = EOS

        prompt_room = MAX_LEN - len(response_ids) - 2  # BOS + SEP
        if prompt_room <= 0:
            continue
        prompt_ids = prompt_ids[-prompt_room:]

        input_ids = [BOS] + prompt_ids + [SEP] + response_ids
        labels = [-100] * (len(prompt_ids) + 2) + response_ids
        if len(input_ids) <= MAX_LEN and any(t != -100 for t in labels):
            yield {
                "input_ids": input_ids,
                "labels": labels,
                "kind": ex.get("kind", "unknown"),
                "doc_id": ex.get("doc_id", ""),
            }


tokenized_examples = list(iter_labeled_examples(examples))
lengths = [len(ex["input_ids"]) for ex in tokenized_examples]
print(f"Labeled examples: {len(tokenized_examples)}")
if lengths:
    print(f"Token length avg={sum(lengths)/len(lengths):.1f}, max={max(lengths)}, context={MAX_LEN}")
    print("Sample labeled example:", tokenized_examples[0])

Labeled examples: 9298863
Token length avg=16.1, max=138, context=512
Sample labeled example: {'input_ids': [1, 311, 279, 346, 688, 463, 754, 984, 334, 408, 2397, 262, 2, 10025, 552, 262, 3], 'labels': [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, 10025, 552, 262, 3], 'kind': 'next_2', 'doc_id': 'SDRW2400002398.1'}


In [ ]:
# Individual examples are padded per batch, not packed into one cross-sample stream.
# This prevents unrelated conversations from attending to each other during supervised response learning.
from collections import Counter

kind_token_counts = Counter()
tokens_per_epoch = 0
for ex in tokenized_examples:
    tokens_per_epoch += len(ex["input_ids"])
    kind_token_counts[ex.get("kind", "unknown")] += len(ex["input_ids"])

print(f"Examples per epoch: {len(tokenized_examples)}")
print(f"Tokens per epoch before dynamic padding: {tokens_per_epoch}")
print("Token mix:", dict(kind_token_counts))

Examples per epoch: 9298863
Tokens per epoch before dynamic padding: 149944111
Token mix: {'next_2': 74856938, 'next_1': 75087173}


# Save Prepared Artifacts


In [ ]:
from pathlib import Path
import gzip
import json
import shutil

PREP_DIR.mkdir(parents=True, exist_ok=True)

for src_name in ['spm16k.model', 'spm16k.vocab']:
    src_path = Path(src_name)
    if not src_path.exists():
        raise FileNotFoundError(f'{src_path} not found. Run the tokenizer cell first.')
    shutil.copy2(src_path, PREP_DIR / src_path.name)
    print(f'Copied: {src_path} -> {PREP_DIR / src_path.name}')

tokenized_path = PREP_DIR / 'tokenized_examples.jsonl.gz'
with gzip.open(tokenized_path, 'wt', encoding='utf-8') as f:
    for row in tokenized_examples:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

pairs_path = PREP_DIR / 'pairs.jsonl.gz'
with gzip.open(pairs_path, 'wt', encoding='utf-8') as f:
    for ex in examples:
        f.write(json.dumps(ex, ensure_ascii=False) + '\n')

manifest = {
    'data_root': str(DATA_ROOT),
    'zip_path': ZIP_PATH,
    'prepared_dir': str(PREP_DIR),
    'model_dir': str(MODEL_DIR),
    'json_files': n_files,
    'documents': n_docs,
    'utterances_seen': n_utterances,
    'usable_units': n_usable_units,
    'training_examples': len(examples),
    'tokenized_examples': len(tokenized_examples),
    'example_mix': dict(kind_counts),
    'tokens_per_epoch_before_padding': tokens_per_epoch,
    'context_length': MAX_LEN,
    'format': 'BOS + prompt + SEP + response + EOS',
    'special_tokens': SPECIALS,
    'tokenizer_model': 'spm16k.model',
    'tokenized_file': tokenized_path.name,
    'pairs_file': pairs_path.name,
}
with open(PREP_DIR / 'data_manifest.json', 'w', encoding='utf-8') as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print('Saved prepared artifacts to:', PREP_DIR)
print('Tokenized rows:', len(tokenized_examples))


Copied: spm16k.model -> /content/drive/MyDrive/text_dataset/manuka/manuka-0527/prepared/spm16k.model
Copied: spm16k.vocab -> /content/drive/MyDrive/text_dataset/manuka/manuka-0527/prepared/spm16k.vocab
Saved prepared artifacts to: /content/drive/MyDrive/text_dataset/manuka/manuka-0527/prepared
Tokenized rows: 9298863
